# ETL VRA/ANAC — 2020
**Contrato v1.1 · ano 2020 · tratamento em Python antes da carga.**

Este notebook substitui a execução apenas em memória por etapas reproduzíveis, auditorias exportadas e uma carga opcional. A fonte bruta e os pipelines dos parceiros não são modificados.

**Execução:** use o ambiente do projeto com `pip install -r requirements.txt`. Abra a partir da raiz ou de `notebooks/etl/`. Os 12 CSVs ficam em `dados/brutos/2020/`.

**Google Colab:** para acessar seu PostgreSQL do Mac em `localhost`, utilize um [ambiente de execução local](https://research.google.com/colaboratory/local-runtimes.html). No Colab hospedado, `localhost` aponta para a máquina do Google, não para seu Mac. Não publique senha nem exponha a porta do banco. O integrante executou a carga pela interface do Colab conectada ao ambiente local. A saída dessa carga e da reexecução está preservada neste notebook.

A etapa de carga vem com `EXECUTAR_CARGA = False`: executar todas as células não carrega dados no banco.

## 1. Preparar o ambiente
Localizamos o checkout e importamos as funções do ETL. A lógica está em um script para que notebook e execução pelo terminal usem as mesmas regras.

In [1]:
from pathlib import Path
import sys

raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/etl_vra_anac_2020.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Abra o notebook dentro do checkout atualizado do projeto.")
sys.path.insert(0, str(raiz / "scripts"))
import etl_vra_anac_2020 as etl
import pandas as pd
from IPython.display import display
print("Contrato:", etl.VERSAO, "| Pandas:", pd.__version__)

Contrato: 1.1 | Pandas: 2.3.3


## 2. Extrair sem modificar a fonte
Exigimos 12 nomes mensais exatos, 20 cabeçalhos e 20 campos por registro CSV. Textos, zeros à esquerda e vazios ficam preservados. `linha_origem` conta registros lógicos a partir de 2; SHA-256 registra a identidade dos arquivos.

In [2]:
entrada = raiz / "dados/brutos/2020"
saida = raiz / "dados/processados/2020"
df_bruta, inventario = etl.extrair(entrada)
display(inventario)
print("Brutos:", len(df_bruta))

,arquivo_origem,registros_brutos,tamanho_bytes,sha256
0,VRA_2020_01.csv,92390,27455960,c64acd1a13397c83861acf5693bf60aa5550acf1fbad08...
1,VRA_2020_02.csv,80919,24326067,ba6a91885d1f1ded54cb46bb714d5a91b26294088a4f82...
2,VRA_2020_03.csv,64523,19232759,e6fe0059e5d784bf7e551bf14aa3e551771c4450dd047c...
3,VRA_2020_04.csv,7617,2113562,c76f5ea70b68896f06335eb54903e2b75cc57ee9b41b6e...
4,VRA_2020_05.csv,9514,2643533,20dfa011cb3423f92674baa629429dfb88b9950fa0a7c4...
5,VRA_2020_06.csv,12800,3578139,d01012074bcb98932f85e0c8ef76e02a2b1c0fc4a52200...
6,VRA_2020_07.csv,19850,5632871,883b2fc9da868733968ae13c03eb6da42834797b8acbfa...
7,VRA_2020_08.csv,24806,7057047,2c187990075bc37701abc3e1a0c4c0f6951cd928b1d34d...
8,VRA_2020_09.csv,30660,8737728,f23a155467ca9424ce6f74cf0acf4182650293aff0fabd...
9,VRA_2020_10.csv,40590,11539155,d12483fe7b950ace85c012c68ded986a9b8281018eaea2...


Brutos: 483545


## 3. Deduplicar, validar e transformar
A igualdade considera apenas os 20 valores originais, antes de remover espaços ou aplicar maiúsculas. Mantemos a primeira ocorrência por arquivo e linha, auditando as excedentes.

Apenas companhia, origem, destino e os dois horários previstos são obrigatórios. Datas reais, assentos e referência inválidos geram anomalias e ficam nulos, sem rejeitar o voo por esse motivo. Códigos e situações recebem maiúsculas; descrições mantêm grafia. Duplicatas que só aparecem após essa padronização são sinalizadas, sem nova exclusão.

In [3]:
quadros, resumo = etl.preparar(df_bruta)
df_transformados = quadros["vra_tratada"]
display(pd.DataFrame([resumo]).T.rename(columns={0: "valor"}))
display(quadros["mensal"])
display(quadros["motivos_rejeicao"].groupby(["campo", "motivo"]).size().rename("ocorrencias"))

,valor
ano_origem,2020
versao_contrato,1.1
brutos,483545
duplicatas_excedentes,124
grupos_duplicados,124
linhas_envolvidas_duplicatas,248
rejeitados,1512
tratados,481909
motivos_rejeicao,3024
intersecao_excedentes_invalidos,0


,arquivo_origem,brutos,duplicatas,rejeitados,tratados
0,VRA_2020_01.csv,92390,4,0,92386
1,VRA_2020_02.csv,80919,30,0,80889
2,VRA_2020_03.csv,64523,90,5,64428
3,VRA_2020_04.csv,7617,0,66,7551
4,VRA_2020_05.csv,9514,0,70,9444
5,VRA_2020_06.csv,12800,0,164,12636
6,VRA_2020_07.csv,19850,0,253,19597
7,VRA_2020_08.csv,24806,0,259,24547
8,VRA_2020_09.csv,30660,0,242,30418
9,VRA_2020_10.csv,40590,0,135,40455


campo             motivo                   
chegada_prevista  CAMPO_OBRIGATORIO_AUSENTE    1512
partida_prevista  CAMPO_OBRIGATORIO_AUSENTE    1512
Name: ocorrencias, dtype: int64

## 4. Conferir métricas e nulos
Diferença = real − previsto, em minutos e com sinal. Atraso = máximo entre diferença e zero. Sem horário real, ambas as métricas são nulas e a flag é falsa. A flag indica diferença absoluta **estritamente maior** que 1.440 minutos; valores suspeitos, inclusive 2202, continuam preservados.

A reconstrução `previsto + diferença = real` testa os cálculos por outro caminho.

In [4]:
for evento in ["partida", "chegada"]:
    real = df_transformados[f"{evento}_real"]
    previsto = df_transformados[f"{evento}_prevista"]
    diferenca = df_transformados[f"diferenca_{evento}_min"]
    atraso = df_transformados[f"atraso_{evento}_min"]
    pd.testing.assert_series_equal(diferenca.isna(), real.isna(), check_names=False)
    pd.testing.assert_series_equal(atraso.isna(), real.isna(), check_names=False)
    conhecidos = real.notna()
    reconstruido = previsto[conhecidos] + pd.to_timedelta(diferenca[conhecidos].astype(float), unit="m")
    pd.testing.assert_series_equal(reconstruido, real[conhecidos], check_names=False)
    assert atraso.dropna().ge(0).all()
    assert atraso[diferenca.lt(0).fillna(False)].eq(0).all()
assert len(df_bruta) == resumo["duplicatas_excedentes"] + resumo["rejeitados"] + len(df_transformados)
display(df_transformados.loc[df_transformados.chegada_real.dt.year.gt(2021),
    etl.CHAVE + ["chegada_prevista", "chegada_real", "diferenca_chegada_min", "chegada_atipica_flag"]])
print("Métricas e reconciliação aprovadas.")

,ano_origem,arquivo_origem,linha_origem,chegada_prevista,chegada_real,diferenca_chegada_min,chegada_atipica_flag
121193,2020,VRA_2020_02.csv,28805,2020-02-04 07:00:00,2022-02-04 06:35:00,1052615.0,True
449136,2020,VRA_2020_12.csv,19916,2020-12-23 03:40:00,2202-12-23 06:16:00,95721276.0,True


Métricas e reconciliação aprovadas.


## 5. Preparar aeroportos e exportar
A dimensão usa uma chave natural ICAO para os papéis origem e destino. `resolucoes_aeroportos_2020.csv` documenta duas descrições conflitantes, com fonte oficial. As descrições originais permanecem nos dados tratados e na auditoria. Qualquer conflito novo bloqueia a estrela e a carga.

A estrela de 2020 é **experimental**: reutiliza o construtor de dimensões e fato do ETL 2019 integrado pelo grupo, sem alterar o arquivo dele. Os IDs desta área de teste não são IDs definitivos do DW compartilhado. A ferramenta `scripts/dim_aeroporto.py` aceita as saídas dos três anos e conserva IDs de um cadastro existente.

Exportamos tratados, rejeitados, motivos, duplicatas, possíveis duplicidades após padronização, anomalias, inventário, contagens e dimensões/fato de teste. Os arquivos grandes ficam em `dados/processados/`, fora do Git.

In [5]:
resultado = etl.executar(entrada, saida,
    extraidos=(df_bruta, inventario), preparados=(quadros, resumo))
assert resultado["summary"]["fontes_preservadas"]
print("Estrela experimental pronta:", resultado["summary"]["estrela_pronta"])
if resultado["dims"] is not None:
    display(resultado["dims"]["dim_aeroporto"].head())
    display(pd.Series(resultado["summary"]["dimensoes"], name="registros"))
print("Saídas:", saida.relative_to(raiz))
print("Carga PostgreSQL: não executada.")
etl.salvar_evidencias(resultado)
print("Evidências pequenas publicadas em resultados/.")

Estrela experimental pronta: True


,id_aeroporto,aeroporto_icao,descricao_aeroporto
0,1,CYUL,MONTRÉAL-PIERRE ELLIOTT TRUDEAU INTERNATIONAL ...
1,2,CYVR,"VANCOUVER INTERNATIONAL AIRPORT - VANCOUVER, B..."
2,3,CYYZ,TORONTO PEARSON INTERNATIONAL AIRPORT - TORONT...
3,4,DGAA,KOTOKA INTERNATIONAL AIRPORT - ACCRA - GANA
4,5,DIAP,PORT BOUET AIRPORT (FELIX HOUPHOUET BOIGNY INT...


dim_tempo              368
dim_companhia_aerea    115
dim_aeroporto          385
dim_situacao_voo         3
dim_justificativa       46
dim_tipo_linha           9
dim_equipamento         62
Name: registros, dtype: int64

Saídas: dados/processados/2020
Carga PostgreSQL: não executada.
Evidências pequenas publicadas em resultados/.


## 6. Carga PostgreSQL — executar somente quando estiver pronto
Primeiro execute `notebooks/configuracao_banco.ipynb`, que só verifica a conexão.

Ao trocar a variável abaixo para `True`, você autoriza a criação das tabelas **somente** no schema experimental `etl_2020` do banco configurado no `.env`. Dimensões entram antes da fato, com chaves estrangeiras. A transação inteira é revertida se ocorrer erro. A mesma execução pode ser repetida sem duplicar registros; um schema de outra execução é recusado. Não há `DROP`, `TRUNCATE` nem substituição automática de dados.

Depois confira o resultado no pgAdmin. A atualização dos dados exige outro schema experimental, não apagar o banco de colegas.

In [7]:
EXECUTAR_CARGA = False
if EXECUTAR_CARGA:
    engine, schema = etl.conectar(raiz / ".env")
    try:
        resultado_carga = etl.carregar(resultado, engine, schema)
        display(resultado_carga)
    finally:
        engine.dispose()
else:
    print("Carga desativada. Nenhuma conexão ou escrita no banco foi feita nesta célula.")

{'status': 'ja_carregada',
 'contagens': {'vra_bruta': 483545,
  'vra_tratada': 481909,
  'vra_duplicatas_excluidas': 124,
  'vra_rejeicoes': 1512,
  'motivos_rejeicao': 3024,
  'vra_anomalias': 0,
  'vra_possiveis_duplicatas_padronizacao': 0,
  'resumo_qualidade': 24,
  'log_carga_arquivo': 12,
  'dim_tempo': 368,
  'dim_companhia_aerea': 115,
  'dim_aeroporto': 385,
  'dim_situacao_voo': 3,
  'dim_justificativa': 46,
  'dim_tipo_linha': 9,
  'dim_equipamento': 62,
  'fato_voo': 481909}}

## 7. Validação SQL da carga
Estas consultas foram reunidas aqui para substituir o arquivo SQL separado. Elas conferem contagens, reconciliação, duplicações, aeroportos órfãos e datas suspeitas **depois da carga**.

A conexão usa uma transação **somente de leitura**. A célula não cria tabelas nem carrega dados. Para repetir as consultas, use `EXECUTAR_VALIDACAO_SQL = True`. O arquivo é salvo com a opção desligada e as saídas da verificação preservadas. A carga continua controlada separadamente por `EXECUTAR_CARGA`.

A evidência adicional em `resultados/validacoes/validacao_postgresql_etl_2020.json` confere também as nove FKs e a consistência das métricas.

In [8]:
EXECUTAR_VALIDACAO_SQL = False

from sqlalchemy import text as sql

def consultar_sql_postgresql(consultas):
    """Executa SELECTs em transação somente de leitura, sem carga ou alteração."""
    engine, schema = etl.conectar(raiz / ".env")
    tabelas = {}
    try:
        with engine.connect() as conn:
            conn.execute(sql("SET TRANSACTION READ ONLY"))
            for titulo, consulta in consultas.items():
                # A conexão valida o nome do schema; permite etl_2020_<sufixo>.
                consulta = consulta.replace("etl_2020.", f'"{schema}".')
                retorno = conn.execute(sql(consulta))
                tabela = pd.DataFrame(retorno.fetchall(), columns=retorno.keys())
                tabelas[titulo] = tabela
                print(titulo)
                display(tabela)
    finally:
        engine.dispose()
    return tabelas

consultas_validacao = {
    'Contagens por camada': """
SELECT 'bruta' AS camada, count(*) FROM etl_2020.vra_bruta
UNION ALL SELECT 'duplicatas', count(*) FROM etl_2020.vra_duplicatas_excluidas
UNION ALL SELECT 'rejeicoes', count(*) FROM etl_2020.vra_rejeicoes
UNION ALL SELECT 'tratada', count(*) FROM etl_2020.vra_tratada
UNION ALL SELECT 'fato', count(*) FROM etl_2020.fato_voo;
    """,
    'Reconciliação': """
SELECT (SELECT count(*) FROM etl_2020.vra_bruta) =
       (SELECT count(*) FROM etl_2020.vra_duplicatas_excluidas) +
       (SELECT count(*) FROM etl_2020.vra_rejeicoes) +
       (SELECT count(*) FROM etl_2020.fato_voo) AS reconciliacao_ok;
    """,
    'Origens duplicadas': """
SELECT count(*) AS origens_duplicadas
FROM (SELECT ano_origem, arquivo_origem, linha_origem FROM etl_2020.fato_voo
      GROUP BY 1,2,3 HAVING count(*) > 1) d;
    """,
    'Aeroportos órfãos': """
SELECT count(*) AS aeroportos_orfaos
FROM etl_2020.fato_voo f
LEFT JOIN etl_2020.dim_aeroporto o ON o.id_aeroporto=f.id_aeroporto_origem
LEFT JOIN etl_2020.dim_aeroporto d ON d.id_aeroporto=f.id_aeroporto_destino
WHERE o.id_aeroporto IS NULL OR d.id_aeroporto IS NULL;
    """,
    'Chegadas em anos suspeitos': """
SELECT chegada_prevista, chegada_real, diferenca_chegada_min, chegada_atipica_flag,
       arquivo_origem, linha_origem
FROM etl_2020.fato_voo WHERE extract(year FROM chegada_real)>2021;
    """,
}

if EXECUTAR_VALIDACAO_SQL:
    validacao_sql = consultar_sql_postgresql(consultas_validacao)
    contagens = validacao_sql["Contagens por camada"].set_index("camada")["count"]
    assert contagens["bruta"] == contagens["duplicatas"] + contagens["rejeicoes"] + contagens["fato"]
    assert contagens["tratada"] == contagens["fato"]
    assert bool(validacao_sql["Reconciliação"].iloc[0, 0])
    assert validacao_sql["Origens duplicadas"].iloc[0, 0] == 0
    assert validacao_sql["Aeroportos órfãos"].iloc[0, 0] == 0
    print("Validação SQL aprovada. Nenhuma tabela foi alterada.")
else:
    print("Consulta ao banco desativada. Ative EXECUTAR_VALIDACAO_SQL após a carga para conferir.")


Contagens por camada
Reconciliação
Origens duplicadas
Aeroportos órfãos
Chegadas em anos suspeitos
Validação SQL aprovada. Nenhuma tabela foi alterada.


,camada,count
0,duplicatas,124
1,rejeicoes,1512
2,fato,481909
3,bruta,483545
4,tratada,481909


,reconciliacao_ok
0,True


,origens_duplicadas
0,0


,aeroportos_orfaos
0,0


,chegada_prevista,chegada_real,diferenca_chegada_min,chegada_atipica_flag,arquivo_origem,linha_origem
0,2020-02-04 07:00:00,2022-02-04 06:35:00,1052615.00,True,VRA_2020_02.csv,28805
1,2020-12-23 03:40:00,2202-12-23 06:16:00,95721276.00,True,VRA_2020_12.csv,19916


## 8. Análise 2 — cancelamentos mensais e efeito dos atípicos
Esta análise é distinta das validações da seção 7. Ela pergunta como o volume elegível e a proporção de cancelamentos variam nos arquivos mensais de 2020. O mês vem do nome do CSV; NÃO INFORMADO permanece categoria separada.

Execute a definição da seção 7 e ative `EXECUTAR_ANALISE_SQL` para consultar o banco, sempre em modo somente de leitura. A consulta de médias apresenta explicitamente o filtro dos horários atípicos, sem excluí-los da base.

In [9]:
EXECUTAR_ANALISE_SQL = False

consultas_analise = {
    'Cancelamentos por mês do arquivo': """
SELECT substring(arquivo_origem FROM 10 FOR 2)::integer AS mes_arquivo,
       count(*) AS voos_elegiveis,
       count(*) FILTER (WHERE situacao_voo='REALIZADO') AS realizados,
       count(*) FILTER (WHERE situacao_voo='CANCELADO') AS cancelados,
       count(*) FILTER (WHERE situacao_voo='NÃO INFORMADO') AS nao_informados,
       round(100.0 * count(*) FILTER (WHERE situacao_voo='CANCELADO') / count(*), 2) AS percentual_cancelados
FROM etl_2020.vra_tratada
GROUP BY mes_arquivo
ORDER BY mes_arquivo;
    """,
    'Efeito das chegadas atípicas na média': """
SELECT count(chegada_real) AS chegadas_medidas,
       count(*) FILTER (WHERE chegada_atipica_flag) AS chegadas_atipicas,
       round(avg(atraso_chegada_min), 2) AS media_atraso_todas_medicoes,
       round(avg(atraso_chegada_min) FILTER (WHERE NOT chegada_atipica_flag), 2) AS media_sem_chegadas_atipicas
FROM etl_2020.vra_tratada;
    """,
}

if EXECUTAR_ANALISE_SQL:
    analise_sql = consultar_sql_postgresql(consultas_analise)
    assert len(analise_sql["Cancelamentos por mês do arquivo"]) == 12
    print("Análise SQL concluída em modo somente de leitura.")
else:
    print("Análise ao banco desativada. Execute a definição da seção 7 e ative EXECUTAR_ANALISE_SQL quando quiser consultar.")


Cancelamentos por mês do arquivo
Efeito das chegadas atípicas na média
Análise SQL concluída em modo somente de leitura.


,mes_arquivo,voos_elegiveis,realizados,cancelados,nao_informados,percentual_cancelados
0,1,92386,91079,1307,0,1.41
1,2,80889,79086,1801,2,2.23
2,3,64428,60220,4203,5,6.52
3,4,7551,6161,1376,14,18.22
4,5,9444,8103,1330,11,14.08
5,6,12636,11332,1294,10,10.24
6,7,19597,18156,1440,1,7.35
7,8,24547,23094,1208,245,4.92
8,9,30418,29192,1225,1,4.03
9,10,40455,38396,2058,1,5.09


,chegadas_medidas,chegadas_atipicas,media_atraso_todas_medicoes,media_sem_chegadas_atipicas
0,460064,331,221.41,8.29


**Interpretação para revisar com a dupla:** abril tem o menor volume elegível e a maior proporção de cancelamentos (18,22%); março tem o maior número absoluto de cancelamentos (4.203). Agosto concentra 245 dos 358 NÃO INFORMADO. Esses padrões não comprovam causas por si só.

A média de atraso de chegada é 221,41 minutos em todas as medições e 8,29 minutos sem as 331 chegadas sinalizadas como atípicas. O filtro só afeta essa consulta. A interpretação completa e os cuidados com o denominador estão no relatório ETL 2020.

## 9. Comparar com o ELT da dupla
A igualdade dos totais não prova igualdade dos registros. Após ambos estarem no mesmo banco de comparação, execute `scripts/comparar_2020.py`: ele compara os 20 atributos tratados, métricas, flags, valores brutos e identidades dos descartes/rejeitados. Não compara IDs técnicos.

O script é somente de leitura. O ELT do parceiro deve ser carregado por seu responsável ou importado para a área de comparação; este notebook não executa o pipeline dele.

**Estado:** carga, validação e reexecução concluídas; notebook executado salvo. Restam comparação por registro com o ELT, revisão cruzada por PR, integração dos aeroportos de 2019/2021 e revisão conjunta dos materiais da entrega.